# 03 — US data and two calendars

Checks for v1.1: the new US symbols, the EUR/USD rate, and what happens when Lisbon and New York calendars are combined.

In [ ]:
import pandas as pd
import plotly.express as px

from portfolio import fx, metrics
from portfolio.config import load_settings
from portfolio.db import get_connection, load_close_prices

settings = load_settings()
conn = get_connection(settings.database_path)
raw = load_close_prices(conn, settings.all_symbols)
conn.close()

pd.DataFrame({
    "days": raw.notna().sum(),
    "first": raw.apply(lambda s: s.first_valid_index().date()),
    "last": raw.apply(lambda s: s.last_valid_index().date()),
    "currency": [settings.currency(s) if s not in settings.fx_symbols.values() else "FX" for s in raw.columns],
})

## 1. EUR/USD rate — plausible range and no spikes

In [ ]:
fx_symbol = settings.fx_symbols["USD"]
eurusd = raw[fx_symbol].dropna()
print(eurusd.describe().round(4))
print("Daily moves above 3%:", (eurusd.pct_change().abs() > 0.03).sum())
px.line(eurusd, title="EUR/USD (USD per 1 EUR)").show()

## 2. Days when only one of the two markets traded

In [ ]:
lisbon = [s for s in settings.tickers + settings.benchmarks if settings.currency(s) == "EUR"]
new_york = [s for s in settings.tickers + settings.benchmarks if settings.currency(s) == "USD"]
lisbon_open = raw[lisbon].notna().any(axis=1)
ny_open = raw[new_york].notna().any(axis=1)

only = pd.DataFrame({
    "Only Lisbon open": (lisbon_open & ~ny_open).groupby(raw.index.year).sum(),
    "Only New York open": (~lisbon_open & ny_open).groupby(raw.index.year).sum(),
})
only

## 3. After conversion to euros and alignment

In [ ]:
fx_columns = {symbol: currency for currency, symbol in settings.fx_symbols.items()}
rates = raw[list(fx_columns)].rename(columns=fx_columns)
assets = raw.drop(columns=list(fx_columns))
aligned = metrics.align_calendars(fx.to_base_currency(assets, settings.currencies, rates))

print("Dates kept:", len(aligned), "| observations per year:", round(metrics.periods_per_year(aligned.index), 1))
# Values carried forward because that market was closed
(assets.reindex(aligned.index).isna()).sum().rename("filled days")

## 4. Extreme daily moves in euros (|return| > 15%)

In [ ]:
returns = metrics.simple_returns(aligned)
big = returns[(returns.abs() > 0.15).any(axis=1)]
big.round(3)

## Conclusions

_Write your findings here:_

- **EUR/USD:** ...
- **Calendars:** how many days a year only one market is open, and what the alignment does with them
- **Extreme moves:** real events or data errors? (Nvidia had stock splits in 2021 and 2024 — adjusted prices should hide them)